In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for updating product sales analysis logic in Databricks
# Purpose: Update bonus eligibility, performance flag, product performance band, and add rep tier per requirements
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script applies updated business logic to sales transactions and rep summary, including error handling, schema validation, and output display. It updates bonus eligibility, performance flag, product performance band, and adds rep_tier, then displays the final output.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import functions as F  
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
)
from pyspark.sql.utils import AnalysisException  

# -- CTE: Load sales_transactions from Unity Catalog
sales_transactions_schema = StructType([
    StructField("transaction_id", IntegerType(), False),
    StructField("rep_id", IntegerType(), False),
    StructField("sales_amount", StringType(), True),
    StructField("bonus_eligibility", StringType(), True),
    StructField("performance_flag", StringType(), True),
    StructField("product_perf_band", StringType(), True),
    StructField("transaction_date", TimestampType(), True)
])

# -- CTE: Load rep_summary from Unity Catalog
rep_summary_schema = StructType([
    StructField("rep_id", IntegerType(), False),
    StructField("total_rep_sales", StringType(), True),
    StructField("rep_tier", StringType(), True),
    StructField("bonus_eligibility", StringType(), True)
])

# -- CTE: Load sales_transactions
def load_sales_transactions():
    """
    Loads sales_transactions from Unity Catalog volume.
    Returns:
        DataFrame: sales_transactions DataFrame
    """
    path = "/Volumes/purgo_databricks/purgo_playground/sales_transactions.csv"
    try:
        df = spark.read.csv(path, header=True, schema=sales_transactions_schema)
        return df
    except Exception as e:
        raise RuntimeError(f"Error loading sales_transactions: {e}")

# -- CTE: Load rep_summary
def load_rep_summary():
    """
    Loads rep_summary from Unity Catalog volume.
    Returns:
        DataFrame: rep_summary DataFrame
    """
    path = "/Volumes/purgo_databricks/purgo_playground/rep_summary.csv"
    try:
        df = spark.read.csv(path, header=True, schema=rep_summary_schema)
        return df
    except Exception as e:
        raise RuntimeError(f"Error loading rep_summary: {e}")

# -- Data validation functions

def validate_required_columns(df, required_cols, df_name):
    """
    Validates that required columns exist in the DataFrame.
    Args:
        df (DataFrame): DataFrame to validate
        required_cols (set): Set of required column names
        df_name (str): DataFrame name for error messages
    Returns:
        None
    """
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s) in {df_name}: {', '.join(missing)}")

def validate_numeric_column(df, col_name, df_name):
    """
    Validates that the column contains only numeric or null values.
    Args:
        df (DataFrame): DataFrame to validate
        col_name (str): Column name
        df_name (str): DataFrame name for error messages
    Returns:
        None
    """
    non_numeric = df.filter(
        (F.col(col_name).isNotNull()) & (F.col(col_name).cast(DoubleType()).isNull())
    ).count()
    if non_numeric > 0:
        raise ValueError(f"{col_name} must be numeric in {df_name}")

def validate_no_null(df, col_name, df_name):
    """
    Validates that the column does not contain null values.
    Args:
        df (DataFrame): DataFrame to validate
        col_name (str): Column name
        df_name (str): DataFrame name for error messages
    Returns:
        None
    """
    null_count = df.filter(F.col(col_name).isNull()).count()
    if null_count > 0:
        raise ValueError(f"{col_name} cannot be null in {df_name}")

def validate_no_duplicate_rep_id(df):
    """
    Validates that there are no duplicate rep_id values in rep_summary.
    Args:
        df (DataFrame): rep_summary DataFrame
    Returns:
        None
    """
    dup_count = df.groupBy("rep_id").count().filter(F.col("count") > 1).count()
    if dup_count > 0:
        raise ValueError("Duplicate rep_id found in rep_summary")

# -- Logic update functions

def update_bonus_eligibility(df):
    """
    Updates bonus_eligibility to 'Yes' if sales_amount > 25000, else keeps existing logic.
    Args:
        df (DataFrame): sales_transactions DataFrame
    Returns:
        DataFrame: Updated DataFrame with bonus_eligibility
    """
    return df.withColumn(
        "bonus_eligibility",
        F.when(F.col("sales_amount").cast(DoubleType()) > 25000, F.lit("Yes"))
         .otherwise(F.col("bonus_eligibility"))
    )

def update_performance_flag(df):
    """
    Updates performance_flag based on sales_amount:
    - High: >9000
    - Medium: <=9000 and >7000
    - Low: <=7000
    Args:
        df (DataFrame): sales_transactions DataFrame
    Returns:
        DataFrame: Updated DataFrame with performance_flag
    """
    return df.withColumn(
        "performance_flag",
        F.when(F.col("sales_amount").cast(DoubleType()) > 9000, F.lit("High"))
         .when((F.col("sales_amount").cast(DoubleType()) <= 9000) & (F.col("sales_amount").cast(DoubleType()) > 7000), F.lit("Medium"))
         .otherwise(F.lit("Low"))
    )

def update_product_perf_band(df):
    """
    Updates product_perf_band based on sales_amount:
    - Excellent: >10000
    - Good: >8000
    - Moderate: >5000
    - Poor: <=5000
    Args:
        df (DataFrame): sales_transactions DataFrame
    Returns:
        DataFrame: Updated DataFrame with product_perf_band
    """
    return df.withColumn(
        "product_perf_band",
        F.when(F.col("sales_amount").cast(DoubleType()) > 10000, F.lit("Excellent"))
         .when(F.col("sales_amount").cast(DoubleType()) > 8000, F.lit("Good"))
         .when(F.col("sales_amount").cast(DoubleType()) > 5000, F.lit("Moderate"))
         .otherwise(F.lit("Poor"))
    )

def update_rep_tier(df):
    """
    Adds/updates rep_tier based on total_rep_sales:
    - Platinum Plus: >45000
    - Platinum: >35000
    - Gold: >25000
    - Silver: <=25000
    Args:
        df (DataFrame): rep_summary DataFrame
    Returns:
        DataFrame: Updated DataFrame with rep_tier
    """
    return df.withColumn(
        "rep_tier",
        F.when(F.col("total_rep_sales").cast(DoubleType()) > 45000, F.lit("Platinum Plus"))
         .when(F.col("total_rep_sales").cast(DoubleType()) > 35000, F.lit("Platinum"))
         .when(F.col("total_rep_sales").cast(DoubleType()) > 25000, F.lit("Gold"))
         .otherwise(F.lit("Silver"))
    )

# -- Main logic

# -- Load data
sales_df = load_sales_transactions()
rep_summary_df = load_rep_summary()

# -- Validate required columns
validate_required_columns(sales_df, {"transaction_id", "rep_id", "sales_amount", "bonus_eligibility", "performance_flag", "product_perf_band", "transaction_date"}, "sales_transactions")
validate_required_columns(rep_summary_df, {"rep_id", "total_rep_sales", "rep_tier", "bonus_eligibility"}, "rep_summary")

# -- Validate numeric columns
validate_numeric_column(sales_df, "sales_amount", "sales_transactions")
validate_numeric_column(rep_summary_df, "total_rep_sales", "rep_summary")

# -- Validate no nulls
validate_no_null(sales_df, "sales_amount", "sales_transactions")
validate_no_null(rep_summary_df, "total_rep_sales", "rep_summary")

# -- Validate no duplicate rep_id
validate_no_duplicate_rep_id(rep_summary_df)

# -- Apply logic updates to sales_transactions
sales_df_updated = update_bonus_eligibility(sales_df)
sales_df_updated = update_performance_flag(sales_df_updated)
sales_df_updated = update_product_perf_band(sales_df_updated)

# -- Apply logic updates to rep_summary
rep_summary_df_updated = update_rep_tier(rep_summary_df)

# -- Join rep_tier to sales transactions for final output
final_output_df = sales_df_updated.join(
    rep_summary_df_updated.select("rep_id", "rep_tier"),
    on="rep_id",
    how="left"
)

# -- Final Output: Display the Output
display(final_output_df)

# spark.stop()  # Do not stop SparkSession in Databricks
